In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from md_Helpers import (
    ProjectPaths,
    SQLiteRunDatabase,
    open_run,
    run_clone_rescale_constant_volume_rate,
    run_clone_rescale_ensemble,
)


# ============================================================
# Configuration — edit these values
# ============================================================

starting_run_id = "20260911174634"

rho_start = 0.65
rho_high = 0.70
rho_low = 0.60

# Applied according to the direction of the density change:
compression_nsteps = 2_000
expansion_nsteps = 2_000

# Fixed-volume NVE evolution after returning to rho_start:
hold_nsteps = 50_000

# Only this box dimension changes during constant-dV/dt ramps:
resize_axis = "x"  # "x", "y", or "z"

# Relaxation-analysis controls:
smoothing_points = 5
equilibrium_tail_fraction = 0.25
equilibrium_tolerance_tail_std = 0.50
equilibrium_sustain_points = 5


# ============================================================
# Validate the supplied starting state
# ============================================================

database = SQLiteRunDatabase(ProjectPaths().database)

records = database.query_thermalizations(
    Run_ID=starting_run_id,
    limit=1,
)

if not records:
    raise ValueError(f"Run_ID {starting_run_id} was not found.")

starting_density = float(records[0]["Density_End"])

if not np.isclose(starting_density, rho_start, rtol=1e-3, atol=1e-6):
    raise ValueError(
        f"Starting run ends at density {starting_density:.8f}, "
        f"not the requested {rho_start:.8f}."
    )

print(
    f"Starting from Run_ID={starting_run_id}, "
    f"rho={starting_density:.8f}"
)


# ============================================================
# Helpers
# ============================================================

def ramp(source_run_id, target_density, nsteps, arm, stage):
    """Run one constant-dV/dt NVE ramp and return its Run_ID."""

    result = run_clone_rescale_constant_volume_rate(
        source_run_id=source_run_id,
        final_density=target_density,
        nsteps=nsteps,
        axis=resize_axis,
        ensemble="NVE",
        notes=(
            f"Two-arm rho={rho_start:g} relaxation experiment; "
            f"arm={arm}; stage={stage}; axis={resize_axis}"
        ),
    )

    run_id = result["run_id"]
    print(
        f"{arm:>4} arm, {stage:<18}: "
        f"rho → {target_density:.3f}, "
        f"Nsteps={nsteps:,}, Run_ID={run_id}"
    )
    return run_id


def fixed_volume_hold(source_run_id, arm):
    """Evolve a returned rho_start state in NVE without changing its box."""

    result = run_clone_rescale_ensemble(
        source_run_id=source_run_id,
        final_density=rho_start,  # Same density means no box change
        nsteps=hold_nsteps,
        ensemble="NVE",
        notes=(
            f"Fixed-volume NVE relaxation after the {arm} arm returned "
            f"to rho={rho_start:g}"
        ),
    )

    run_id = result["run_id"]
    print(
        f"{arm:>4} arm, fixed-volume hold: "
        f"Nsteps={hold_nsteps:,}, Run_ID={run_id}"
    )
    return run_id


# ============================================================
# Construct the two independent arms
# ============================================================

run_ids = {
    "high": {},
    "low": {},
}

# High arm: 0.65 -> 0.70 -> 0.65
run_ids["high"]["outward"] = ramp(
    starting_run_id,
    rho_high,
    compression_nsteps,
    arm="high",
    stage="0.65 -> 0.70",
)

run_ids["high"]["returned"] = ramp(
    run_ids["high"]["outward"],
    rho_start,
    expansion_nsteps,
    arm="high",
    stage="0.70 -> 0.65",
)

# Low arm: 0.65 -> 0.60 -> 0.65
run_ids["low"]["outward"] = ramp(
    starting_run_id,
    rho_low,
    expansion_nsteps,
    arm="low",
    stage="0.65 -> 0.60",
)

run_ids["low"]["returned"] = ramp(
    run_ids["low"]["outward"],
    rho_start,
    compression_nsteps,
    arm="low",
    stage="0.60 -> 0.65",
)


# ============================================================
# Fixed-volume NVE relaxation runs
# ============================================================

run_ids["high"]["hold"] = fixed_volume_hold(
    run_ids["high"]["returned"],
    arm="high",
)

run_ids["low"]["hold"] = fixed_volume_hold(
    run_ids["low"]["returned"],
    arm="low",
)

print("\nGenerated runs:")
display(pd.DataFrame(run_ids).T)


# ============================================================
# Load and verify the fixed-volume runs
# ============================================================

hold_logs = {}

for arm in ("high", "low"):
    logs = open_run(run_ids[arm]["hold"]).logs_dataframe().copy()

    logs["potential_energy_per_particle"] = (
        logs["potential_energy"] / logs["num_particles"]
    )

    mean_volume = float(logs["volume"].mean())
    relative_volume_range = float(
        np.ptp(logs["volume"].to_numpy(dtype=float)) / mean_volume
    )

    if relative_volume_range > 1e-10:
        raise RuntimeError(
            f"{arm} hold is not constant volume: "
            f"relative volume range={relative_volume_range:.3e}"
        )

    hold_logs[arm] = logs

    print(
        f"{arm.capitalize()} hold: "
        f"rho={logs['density'].iloc[-1]:.8f}, "
        f"relative volume range={relative_volume_range:.3e}"
    )


# ============================================================
# Operational relaxation-time estimate
#
# A state is labeled equilibrated when rolling averages of
# temperature, pressure, and PE/N all remain close to their
# late-time values for equilibrium_sustain_points samples.
# ============================================================

observables = [
    "kinetic_temperature",
    "pressure",
    "potential_energy_per_particle",
]

def estimate_relaxation(logs):
    n = len(logs)

    if n < 2 * smoothing_points:
        raise ValueError(
            f"Only {n} log samples are available. Increase hold_nsteps "
            "or use a source run with a shorter inherited log period."
        )

    tail_points = max(
        3 * smoothing_points,
        int(np.ceil(equilibrium_tail_fraction * n)),
    )
    tail_points = min(tail_points, n)
    tail = logs.iloc[-tail_points:]

    equilibrated = pd.Series(True, index=logs.index)
    rolling_curves = {}
    reference = {}

    for column in observables:
        rolling = (
            logs[column]
            .rolling(smoothing_points, min_periods=smoothing_points)
            .mean()
        )

        tail_mean = float(tail[column].mean())
        tail_std = float(tail[column].std(ddof=1))

        numerical_floor = np.finfo(float).eps * max(1.0, abs(tail_mean))
        tolerance = max(
            equilibrium_tolerance_tail_std * tail_std,
            numerical_floor,
        )

        rolling_curves[column] = rolling
        reference[column] = {
            "mean": tail_mean,
            "std": tail_std,
            "tolerance": tolerance,
        }

        equilibrated &= (rolling - tail_mean).abs() <= tolerance

    sustained = (
        equilibrated.astype(int)
        .rolling(
            equilibrium_sustain_points,
            min_periods=equilibrium_sustain_points,
        )
        .sum()
        .eq(equilibrium_sustain_points)
    )

    matches = np.flatnonzero(sustained.to_numpy())

    if len(matches) == 0:
        equilibrium_index = None
        equilibrium_time = np.nan
    else:
        # Start of the first sustained interval rather than its endpoint
        equilibrium_index = max(
            0,
            int(matches[0]) - equilibrium_sustain_points + 1,
        )
        equilibrium_time = float(
            logs["this_lj_time"].iloc[equilibrium_index]
        )

    return equilibrium_time, equilibrium_index, rolling_curves, reference


relaxation_results = {
    arm: estimate_relaxation(logs)
    for arm, logs in hold_logs.items()
}


# ============================================================
# Plot the fixed-volume NVE relaxation
# ============================================================

plot_specs = [
    ("kinetic_temperature", "Kinetic temperature"),
    ("pressure", "Pressure"),
    ("potential_energy_per_particle", "Potential energy / particle"),
]

colors = {
    "high": "tab:red",
    "low": "tab:blue",
}

labels = {
    "high": r"Returned from $\rho=0.70$",
    "low": r"Returned from $\rho=0.60$",
}

fig, axes = plt.subplots(
    3,
    1,
    figsize=(12, 11),
    sharex=True,
)

for arm in ("high", "low"):
    logs = hold_logs[arm]
    equilibrium_time, _, rolling_curves, reference = (
        relaxation_results[arm]
    )

    time = logs["this_lj_time"].to_numpy(dtype=float)

    for ax, (column, ylabel) in zip(axes, plot_specs):
        color = colors[arm]

        ax.plot(
            time,
            logs[column],
            color=color,
            alpha=0.18,
            linewidth=0.8,
        )
        ax.plot(
            time,
            rolling_curves[column],
            color=color,
            linewidth=2,
            label=labels[arm],
        )

        tail_mean = reference[column]["mean"]
        tolerance = reference[column]["tolerance"]

        ax.axhspan(
            tail_mean - tolerance,
            tail_mean + tolerance,
            color=color,
            alpha=0.06,
        )
        ax.set_ylabel(ylabel)
        ax.grid(alpha=0.3)

    if np.isfinite(equilibrium_time):
        for ax in axes:
            ax.axvline(
                equilibrium_time,
                color=colors[arm],
                linestyle="--",
                linewidth=1.3,
                alpha=0.9,
            )

axes[-1].set_xlabel("Time since start of fixed-volume NVE hold")
axes[0].legend()

fig.suptitle(
    r"Relaxation after the two arms return to $\rho=0.65$"
    "\nDashed lines show the operational relaxation-time estimates",
    fontsize=14,
)

plt.tight_layout()
plt.show()


# ============================================================
# Report estimated relaxation times
# ============================================================

summary_rows = []

for arm in ("high", "low"):
    equilibrium_time = relaxation_results[arm][0]

    summary_rows.append({
        "arm": arm,
        "history": (
            f"{rho_start:.2f} -> "
            f"{rho_high if arm == 'high' else rho_low:.2f} -> "
            f"{rho_start:.2f}"
        ),
        "returned_run_id": run_ids[arm]["returned"],
        "hold_run_id": run_ids[arm]["hold"],
        "estimated_relaxation_LJ_time": equilibrium_time,
        "estimated_relaxation_step": (
            np.nan
            if not np.isfinite(equilibrium_time)
            else int(
                hold_logs[arm].loc[
                    hold_logs[arm]["this_lj_time"] >= equilibrium_time,
                    "run_step",
                ].iloc[0]
            )
        ),
    })

relaxation_summary = pd.DataFrame(summary_rows)
display(relaxation_summary)

Starting from Run_ID=20260911174634, rho=0.65000000
high arm, 0.65 -> 0.70      : rho → 0.700, Nsteps=2,000, Run_ID=20260914170511
